Dataset:
- Kvasir-SEG: download thủ công (https://datasets.simula.no/kvasir-seg/)
- Kvasir-v2: download thủ công (https://datasets.simula.no/kvasir/)

In [1]:
from pathlib import Path
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

DATA = Path("..") / "data"
VQA_IMG_DIR = DATA / "kvasir-vqa" / "images"
VQA_IMG_DIR.mkdir(parents=True, exist_ok=True)

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Load dataset

In [2]:
ds_img = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")
print(ds_img)
print(f"Total rows: {len(ds_img)}")

Dataset({
    features: ['image', 'source', 'question', 'answer', 'img_id'],
    num_rows: 58849
})
Total rows: 58849


### 2. Lưu ảnh theo img_id (không trùng ảnh)

In [3]:
saved = 0
skipped = 0

for row in tqdm(ds_img, desc="Saving images"):
    img_id = row["img_id"]
    out_path = VQA_IMG_DIR / f"{img_id}.jpg"

    if out_path.exists():
        skipped += 1
        continue

    img = row["image"]  # PIL Image
    img.convert("RGB").save(out_path, format="JPEG")
    saved += 1

print(f"Saved: {saved} | Skipped (already exist): {skipped}")
print(f"Total unique images on disk: {len(list(VQA_IMG_DIR.glob('*.jpg')))}")

Saving images: 100%|██████████| 58849/58849 [07:28<00:00, 131.26it/s] 

Saved: 6500 | Skipped (already exist): 52349
Total unique images on disk: 6500


In [4]:
unique_ids = set(ds_img["img_id"])
on_disk = set(p.stem for p in VQA_IMG_DIR.glob("*.jpg"))

missing = unique_ids - on_disk
print(f"Unique img_id in dataset: {len(unique_ids)}")
print(f"Images on disk: {len(on_disk)}")
print(f"Missing: {len(missing)}")
if missing:
    print("Sample missing:", list(missing)[:5])

Unique img_id in dataset: 6500
Images on disk: 6500
Missing: 0


### 3. Kiểm tra Kvasir-VQA-x1 có dùng hết ảnh từ dataset Kvasir-VQA

In [5]:
ds_x1 = load_dataset("SimulaMet/Kvasir-VQA-x1")
ds_host = load_dataset("SimulaMet-HOST/Kvasir-VQA", split="raw")

x1_ids = set(ds_x1["train"]["img_id"]) | set(ds_x1["test"]["img_id"])
host_ids = set(ds_host["img_id"])

only_in_x1 = x1_ids - host_ids
only_in_host = host_ids - x1_ids

print(f"img_id unique trong x1   : {len(x1_ids)}")
print(f"img_id unique trong host : {len(host_ids)}")
print(f"Có trong x1 nhưng không có ảnh trong host: {len(only_in_x1)}")
print(f"Có trong host nhưng x1 không reference   : {len(only_in_host)}")
if only_in_host:
    print(f"Ảnh dư: {only_in_host}")

img_id unique trong x1   : 6449
img_id unique trong host : 6500
Có trong x1 nhưng không có ảnh trong host: 0
Có trong host nhưng x1 không reference   : 51
Ảnh dư: {'clb0kvxun8zkw074y4bhk38pl', 'clb0lbwzhdp60086u1c4nfcnk', 'clb0lbwzidp74086u7h4p1jbf', 'clb0lbwzudpmg086uewzqdwqu', 'clb0lbx00dptk086ugc62aqid', 'clb0kvxun8zlg074y12zv2jvt', 'cla820gles4br071u2ms2d20c', 'cla820glas44z071uewo77wmu', 'clb0lbx01dpv4086u0m8405q9', 'clb0lbx07dq6s086ufm8v2qnu', 'clb0lbwz3doq4086u0p0w5jzm', 'clb0lbwzjdp9c086uhdzgdzwy', 'clb0lbwzudplo086u5pnr0c4c', 'cl8k2u1s51guf08327wxwftl6', 'clb0kvxun8zl4074y7b8e8fqq', 'clb0kvxun8zlk074y7drl4a6h', 'clb0lbwzudplk086ubfwge9ki', 'clb0lbx02dpwg086uenmd1sqy', 'clb0lbwzrdpj4086u15an5jfr', 'clb0lbwzkdp9k086u1icv3f9v', 'cl8k2u1rm1gcz083207yx4s74', 'clb0lbwzhdp68086u1g8ocw9m', 'cl8k2u1s51gur083261cs3cza', 'clb0lbx00dpt8086uc3i84db8', 'clb0kvxun8zls074yh71ngdy1', 'cl8k2u1pr1dzb08323zngbla5', 'clb0lbwz7douo086u72nrh0tg', 'clb0lbx1ndqq4086ugrbm0k5o', 'cla820glcs47j071uccmo19

- Bộ dataset Kvasir-VQA-x1 sử dụng lại ảnh của bộ dataset Kvasir-VQA.
- Nhưng Kvasir-VQA-x1 chỉ sử dụng 6449 trong data set Kvasir-VQA.